In [ ]:
import pandas as pd
import numpy as np  
# from sodapy import Socrata
from datetime import datetime
import os
# from tqdm import tqdm
import sys
from pathlib import Path
import matplotlib.pyplot as plt
# import seaborn as sns


pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", 100)
pd.set_option('display.float_format', '{:.4f}'.format)

# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))
import utils

path = os.getcwd()
print(path)

path_repo = "C:/Users/PC/Desktop/tesis/"
path_data = "C:/Users/PC/Desktop/tesis/data/"

In [ ]:
qwe

## Tasas colocación V1

In [ ]:
# Tabla semanal
tasascol = pd.read_csv(path_repo + "/data/2022.csv", decimal = ",", thousands = ".", encoding='utf-8')
tasascol.columns = [x.lower().replace(" ","_") for x in tasascol.columns]
tasascol.rename(columns = {"tasa_efectiva_promedio_ponderada":"tasa_ponderada", "montos_desembolsados":"monto", "numero_de_creditos_desembolsados":"num_creditos", 
                           "sum_numero_de_creditos":"num_creditos", "producto_de_crédito":"producto", "plazo_de_crédito":"plazo", "nombre_entidad":"banco"}, inplace=True)
tasascol.drop(columns = ["tipo_entidad", "codigo_entidad"], inplace =True)
tasascol.sample(5)

In [ ]:
cols_cat = ['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_persona', 'sexo', 'tamaño_de_empresa', 'tipo_de_crédito', 'tipo_de_garantía',
            'producto', 'plazo', 'grupo_etnico', 'antiguedad_de_la_empresa', 'tipo_de_tasa', 'clase_deudor', 'codigo_ciiu', 'codigo_municipio',
            'rango_monto_desembolsado']
cols_cont = ['tasa_ponderada', 'margen_adicional', 'monto', 'num_obl']

tasascol["fecha_corte"] = pd.to_datetime(tasascol["fecha_corte"], dayfirst=True, errors='coerce')

tasascol["tasa_ponderada"] = tasascol["tasa_ponderada"].astype(float)
tasascol["num_creditos"] = tasascol["num_creditos"].astype(int)
tasascol["fecha_corte"] = pd.to_datetime(tasascol["fecha_corte"])
tasascol["anomes"] = tasascol["fecha_corte"].dt.strftime("%Y-%m")

# tasascol = tasascol[~tasascol["banco"].isin(["Fondo Nacional del Ahorro", "La Hipotecaria", "Bancoldex", "Confiar",
#                                         "IRIS C.F.", "Bancamía S.A.", "Bancien", "Mibanco S.A.", "Banco Contactar",
#                                         'Tuya','Crezcamos','Credifamilia','Financiera Juriscoop C.F.','Rappipay',
#                                         'Banco GNB Sudameris','Banco Popular','Coopcentral','Banco Pichincha S.A.','Nu Colombia C.F.',
#                                         "Caja de Vivienda Militar y de Polícia"])] # Excluidos por tasas atipicos
                                        
# tasascol = tasascol[tasascol["tasa_ponderada"] <= 50]
# tasascol = tasascol[tasascol["num_creditos"] >= 2]
# tasascol = tasascol[(tasascol["rango_monto_desembolsado"].notna())] # tienen nulos antes de 202309

tasascol ["banco"] = tasascol["banco"].str.replace("Compañía De Financiamiento","CF").str.replace("Cooperativa Financiera","CF")
tasascol["plazo"] = tasascol["plazo"].replace({'Más de 1 año y hasta 3 años':"(1-3]", 'Más de 3 años y hasta 5 años':"(3-5]",
                                                'Más de 5 años y hasta 7 años':"(5-7]",
                                                'Más de 7 años y hasta 10 años':"(7-10]", 'Más de 10 años':"(10-]",
                                                "Hasta 6 meses":"(0-1]", "Más de 6 meses y hasta 1 año":"(0-1]",
                                                'Hasta 30 días':"(0-1]", 'Entre 31 días y 1 año':"(0-1]"})

tasascol["prom_monto_porobl"] = tasascol["monto"]/tasascol["num_creditos"]

print(tasascol["fecha_corte"].min(), "|", tasascol["fecha_corte"].max())

print(tasascol.shape)
print("Bancos:", tasascol["banco"].nunique())

tasascol.sample(7)

In [ ]:
tasascol[tasascol["anomes"] == "2022-08"]["fecha_corte"].value_counts()

In [ ]:
utils.vc(tasascol["tipo_de_crédito"])

In [ ]:
utils.vc(tasascol["tipo_de_garantía"])

In [ ]:
tasascol.isnull().mean()

In [ ]:
# Sample
tmp = tasascol.drop(columns = ['margen_adicional', 'grupo_etnico', 'antiguedad_de_la_empresa', 'clase_deudor', 'codigo_ciiu',
       'codigo_municipio', 'anomes', 'prom_monto_porobl']).sample(100)
tmp.to_csv(path_data+'sample.csv', index = False)
tmp.sample(5)

In [ ]:
print(np.percentile(tasascol["tasa_ponderada"], 0.05))
print(np.percentile(tasascol["tasa_ponderada"], 2.25))
print(np.percentile(tasascol["tasa_ponderada"], 5))
print(np.percentile(tasascol["tasa_ponderada"], 50))
print(np.percentile(tasascol["tasa_ponderada"], 95))
print(np.percentile(tasascol["tasa_ponderada"], 97.5))
print(np.percentile(tasascol["tasa_ponderada"], 99.5))

In [ ]:
print(tasascol[tasascol["tasa_ponderada"] <= 10].shape)
print(tasascol[tasascol["tasa_ponderada"] >= 40].shape)

In [ ]:
# ax = tasascol[tasascol["fecha_corte"] >= "2025-01-01"]
ax = tasascol
ax = ax.groupby("producto").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["producto", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
# ax = tasascol[tasascol["fecha_corte"] >= "2025-01-01"]
ax = tasascol
ax = ax.groupby("banco").agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["banco", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
ax = ax.sort_values("num_creditos", ascending=False)#.head(30)
ax

In [ ]:
tasascol["anomes"].value_counts(dropna = False, normalize =True)
ax = tasascol
ax = ax.groupby(["anomes"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":["mean", "median", "min", "max", "std"]}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax.columns = ["mes", "monto", "num_creditos", "tasa_mean", "tasa_median", "tasa_min", "tasa_max", "tasa_std"]
ax["tasa_cv"] = ax["tasa_std"]/ax["tasa_mean"].mean()*100
ax["%monto"] = ax["monto"]/ax["monto"].sum()*100
ax["%num_obl"] = ax["num_creditos"]/ax["num_creditos"].sum()*100
ax["prom_monto"] = ax["monto"]/ax["num_creditos"]
#ax.sort_values("monto", ascending=False).head(40)
ax

In [ ]:
ax = tasascol
ax = ax.groupby(["anomes", "plazo", "rango_desembolso"]).agg({"monto":"sum", "num_creditos":"sum", "tasa_ponderada":"mean"}).reset_index()#.sort_values("num_creditos", ascending=False)#.head(30)
ax["monto"] = ax["monto"]/1000000
ax["prom_monto_porobl"] = ax["monto"]/ax["num_creditos"] # millones

#ax.sort_values("monto", ascending=False).head(40)
tasascol = ax
tasascol

## How to group?

In [ ]:
print(tasascol.shape)
tasascolgr = tasascol.copy()
tasascolgr[cols_cat] = tasascolgr[cols_cat].fillna("NULL")
tasascolgr["tasaXmonto"]  = tasascolgr['tasa_ponderada']*tasascolgr['monto']
tasascolgr = tasascolgr.groupby(['nombre_tipo_entidad', 'banco', 'fecha_corte', 'tipo_de_crédito', 'tipo_de_garantía', 'tipo_de_persona', #'sexo', 'tamaño_de_empresa',  'producto',
                                'plazo', #'grupo_etnico', 'antiguedad_de_la_empresa','tipo_de_tasa', 
                                'rango_monto_desembolsado', #'clase_deudor',
                                #'codigo_ciiu', 'codigo_municipio', 'anomes', 'prom_monto_porobl'
                                ]).agg({"monto":"sum", "tasaXmonto":"sum", 'num_creditos':"sum"}).reset_index()
print(tasascolgr.shape)
tasascolgr.sample(5)

In [ ]:
601806204910.0000/30926219000.0000